In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch" # tells Keras to use PyTorch

import torch
import keras
from sentence_transformers import SentenceTransformer
import pandas as pd
from pathlib import Path
m = SentenceTransformer("all-MiniLM-L6-v2") # SBERT Model

root = Path(r"C:\Users\nicky\aims\aims\data")
dirs = sorted({p.parent for p in root.rglob("response.parquet")})

resp = pd.concat([pd.read_parquet(d / "response.parquet").assign(bench=d.name) for d in dirs], ignore_index=True)
items = pd.concat([pd.read_parquet(d / "items.parquet").assign(bench=d.name) for d in dirs], ignore_index=True)
subj = pd.concat([pd.read_parquet(d / "subjects.parquet").assign(bench=d.name) for d in dirs], ignore_index=True)

resp.head()

c:\Users\nicky\aims\aims\starting_kit\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20204.53it/s]


,subject_id,item_id,response_id,benchmark_id,trial,test_condition,interactors,response,bench
0,8413d1f86426bfa2,25fe37ba23449698,e650e6c528f66831,matharena,1,NaN,NaN,1.0,matharena
1,8413d1f86426bfa2,25fe37ba23449698,7a5ed2968df1a6f4,matharena,2,NaN,NaN,1.0,matharena
2,8413d1f86426bfa2,25fe37ba23449698,9a1dab89c874baa8,matharena,3,NaN,NaN,1.0,matharena
3,8413d1f86426bfa2,25fe37ba23449698,5c7f5ce897fcf7c7,matharena,4,NaN,NaN,1.0,matharena
4,8413d1f86426bfa2,d6fd91839534dc84,ba770e09cddc74a8,matharena,1,NaN,NaN,0.0,matharena


In [ ]:
import numpy as np

# The average squared difference between predictions and outcomes. np.sarray converts pandas columns to NumPy arrays so the subtraction works element by element.
def brier(y_true, y_pred):
    return np.mean((np.asarray(y_pred) - np.asarray(y_true)) ** 2)

# Groups by rows (AI System, Benchmark) pair and calculates the Brier score for each pair.
# The lambda function runs once per group, and the mean averages those pair scores equally so all row sizes are counted equally. Which matches the AIMS eval metric.
def score(df, pred_col="pred"):
    """Brier per subject/benchmark pair, then a plain average."""
    per_pair = df.groupby(["subject_id", "benchmark_id"]).apply(
        lambda g: brier(g["y"], g[pred_col]))
    return per_pair.mean()

# Same labels as the baseline notebook, so the NN and baselines are scored on the same targets
resp = resp.dropna(subset=["response"])
resp["y"] = (resp["response"] > 0.5).astype(int)

### My Whiteboard Implementation: Keras 3 on PyTorch

**IN:** Turning each tasks text into a vector with SBERT and feed it to the network : took eacg unique item once (12,632 items) then read its content column and ran the texts through SBERT (all-MiniLM-L6-v2). Each text became 384 numbers capturing its meaning. Vectors saved to item_emb.npy and their order to item_emb_ids.csv. Worth noting, long texts are truncated since MiniLM reads at most 256 tokens. Also worth noting, I have the SBERT vectors frozen (trainable=false). 

**IN:** Instead of hash(subject_id) I used normalized_name

**Hidden Layer:** Dense(64,ReLU), the concatenated 384 items + 16 name (400 inputs) are fed into 64 neurons. ReLU passes positive values through and zeros out negatives which allows the network to learn non-linear patterns. Im not sure if 64 neurons is a choice that can be improved upon.

**Output Sigmoid:** Dense(1, sigmoid)

**Backprop:** The network trains on raw response values, using MSE loss. Predictions are scored against rounded 0/1 labels (response > 0.5) to match the AIMS binary outcomes. Training on rounded labels instead gave 0.385 vs 0.344. 


Compare the output to response and backprop the error : I use raw value threshold, the loss is the mse between prediction and y which is the brier score, model.fit handles backprop in Keras with the optimizer (adam over SGD, AdamW, RMSprop)(learning rate 0.001) updating weights after each batch of 1024 rows. Worth noting: I binarized response because my earlier findings showed mmdocrag's response is a rubric score and matharena contains some partial credit, so the labels are the only difference, optimizer/batch-size/early-stopping are not mentioned in whiteboard, if it hasnt improved after 3 epochs training stops and the weights rolls back to the best epoch, max 20.







In [ ]:
#Load the saved embeddings
uniq = items.drop_duplicates("item_id").reset_index(drop=True) #rebuilds the list of unique items in the same order as when they were embedded
emb = np.load("item_emb.npy")  
saved_ids = pd.read_csv("item_emb_ids.csv")["item_id"] 
assert (saved_ids.values == uniq["item_id"].values).all(), "item order changed: re-embed" #an error check that compares the saved id order with the current one item by item. If anything differs an error message is posted. 
print(emb.shape)

(12632, 384)


In [ ]:
## encode subjects

#creates one name per subject_id looked up by set_index
names = subj.drop_duplicates("subject_id").set_index("subject_id")["normalized_name"]
names = names.astype("string").str.lower().str.strip() #treats lower and upper case the same
names = names.mask(names == "") #empty strings are treated as missing values

vocab = {n: i + 1 for i, n in enumerate(sorted(names.dropna().unique()))} #vocab is a dictionary mapping each distinct name to a number from 1 to 132. Enumerate counts from 0 so 0 i+1 keeps 0 reserved for unknown. having it sorted makes the numbering reporducible. 
name_idx = names.map(vocab).fillna(0).astype(int) #replaces each name with its number, .fillna(0) gives missing names 0. 
print(f"{len(vocab)} distinct model names | {(name_idx == 0).mean():.0%} of subjects have no name")


item_pos = pd.Series(range(len(uniq)), index=uniq["item_id"]) #finding item_id row in emb 
# the map calls attach to every response row the row number of its items vector and the number of its AI systems name. These two numbers are what the NN recieves as input. 
resp["item_pos"] = resp["item_id"].map(item_pos) 
resp["name_idx"] = resp["subject_id"].map(name_idx)

print("rows missing an item embedding:", resp["item_pos"].isna().sum())
print("rows missing a subject id:", resp["name_idx"].isna().sum())
print(f"rows with unknown name (idx 0): {(resp['name_idx'] == 0).mean():.0%}")

132 distinct model names | 18% of subjects have no name
rows missing an item embedding: 0
rows missing a subject id: 0
rows with unknown name (idx 0): 6%


In [ ]:
import numpy as np
import pandas as pd
import keras

emb = np.load("item_emb.npy") # (12632, 384) SBERT vectors, row i = uniq item i
n_names = len(vocab) + 1 # +1 for unknown (0)

def build_model():

    #two entry points, shape=() means a single number, not a vector.
    item_in = keras.Input(shape=(), dtype="int32", name="item_pos")
    name_in = keras.Input(shape=(), dtype="int32", name="name_idx")

    # embedding layer look up table for 12,632 rows of 384 numbers. trainable=False means the vectors are fixed (frozen), not updated during training.
    sbert = keras.layers.Embedding(emb.shape[0], emb.shape[1], trainable=False, name="sbert_lookup")
    item_vec = sbert(item_in)
    # a second embedding layer with 133 rows (132 names + 1 unknown) of 16 numbers each. This layer is trainable, the values start random and are updated during training into a learned description of each AI System. 
    name_vec = keras.layers.Embedding(n_names, 16, name="name_emb")(name_in)

    x = keras.layers.Concatenate()([item_vec, name_vec]) # joins the two vectors end to end into a single vector of 384 + 16 = 400 numbers.
    x = keras.layers.Dense(64, activation="relu")(x) # hidden layer
    out = keras.layers.Dense(1, activation="sigmoid")(x) # P(correct)

    model = keras.Model([item_in, name_in], out) # connects inputs to output
    sbert.set_weights([emb]) # copies the saved BERT vectors into the frozen table. runs after model is built so the layer exists.
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse") # sets training rules: Adam optimizer with learning rate 0.001, mean squared error loss function.
    return model

# this function puts a dataframes two input columns as Numpy arrays in a dictionary keyed by the input names (what Keras expects). The names are passed seperately so run_fold can substitute adjusted names for held-out rows. 
def inputs(df, names):
    return {"item_pos": df["item_pos"].to_numpy("int32"), "name_idx": names.to_numpy("int32")}


def run_fold(held_out, epochs=20, seed=0, verbose=2):

    # seed fixed to make random choices reproducible. 
    keras.utils.set_random_seed(seed)

    # leave one benchmark out split. train on every other benchmark, test on the held-out one. (.copy() avoids pandas warnings when adding a pred column to test)
    train = resp[resp["benchmark_id"] != held_out]
    test = resp[resp["benchmark_id"] == held_out].copy()


    # the names that appear in training. For held out rows, (where(condition,0)) keeps the name ID if the name was seen and replaces it with 0 otherwise, so unseen names use the trained "unknown" vectir instead of random, untrained values.
    seen = set(train["name_idx"])
    test_names = test["name_idx"].where(test["name_idx"].isin(seen), 0)

    # picks 10% of training items at random, is_stop marks every row whose item was picked, ~ is not, so fit_df is the rows the network learns from and stop_df is the rows used only to decide when to stop. splitting by item keeps all trials of tasks together on one side. 
    rng = np.random.default_rng(seed)
    tr_items = train["item_id"].unique()
    stop_items = rng.choice(tr_items, size=len(tr_items) // 10, replace=False)
    is_stop = train["item_id"].isin(stop_items)
    fit_df, stop_df = train[~is_stop], train[is_stop]

    model = build_model()

    ## This is training
    # the targets are the raw response values (option B). the commented-out lines above it are the old rounded-label version. 
    # each epoch is one pass through fit_df in batches of 1024 rows. For each batch, Keras computes predicitons/MSE/gradients (backprop) and Adam updates the weights.
    # after each epoch, Keras computes the MSE on stop_df (val_loss). EarlyStopping ends training if that hasn't improved for 3 epochs and restores the best epochs weights.
    # hist.history records loss and val_loss for every epoch. 
    hist = model.fit(
        #inputs(fit_df, fit_df["name_idx"]), fit_df["y"].to_numpy("float32"),
        #validation_data=(inputs(stop_df, stop_df["name_idx"]), stop_df["y"].to_numpy("float32")),
        inputs(fit_df, fit_df["name_idx"]), fit_df["response"].to_numpy("float32"), 
        validation_data=(inputs(stop_df, stop_df["name_idx"]), stop_df["response"].to_numpy("float32")), 
        epochs=epochs, batch_size=1024, verbose=verbose,
        callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)])

    # this predicts on the held-out benchmark. .ravel() flattens Keras output chape (n,1) into a plain list of n values. 
    # This returns the pair-weighted Brier score (against rounded y) and the training history as a table. 
    test["pred"] = model.predict(inputs(test, test_names), batch_size=4096, verbose=0).ravel()
    return score(test), pd.DataFrame(hist.history)

# For each bennchmark, run a full fold with it held out and keep element 0 of the result (score). It trains six seperate models from scratch. 
nn_results = {b: run_fold(b, verbose=0)[0] for b in sorted(resp["benchmark_id"].unique())}
nn = pd.Series(nn_results, name="neural_net").round(4)
display(nn)
print("avg (6 folds):", nn.mean().round(4), "| avg excl. mmdocrag:", nn.drop("mmdocrag").mean().round(4))

matharena            0.2892
mmdocrag             0.4122
multi_swebench       0.4069
real_webagents       0.3229
researchcodebench    0.3470
swe_rebench          0.2836
Name: neural_net, dtype: float64

avg (6 folds): 0.3436 | avg excl. mmdocrag: 0.3299


 the neural net performed the worst so far with the labels rounded. 0.385 avg brier using leave-one-benchmark-out. With the raw whiteboard listed approach I got 0.344 whch is better but still worse than cointoss, excluding mmdocrag for a robustness check, since it is different from the others being a rubric score, gets 0.330. Its still learning benchmark-specific patterns likely because of the large difference in embeddings of text, math,code so its lumping all of one topic in together regardless of difficulty. 

potential future paths: 

- reducing network confidence? Unless the evidence is strong stay close to 0.5 (the strongest so far) or try dropout/smaller network

- Try and isolate what input is causing the damage, item-only/name-only versions?

- Within the five training benchmarks, hold one out for early stopping, so the network gets rewarded for generalizing across benchmarks. Early stop on a held out benchmark instead of held out items

- Note: Test time adaptation might end up being the most important thing